# Product Clustering: Cross-Supermarket Matching

Pipeline to cluster grocery products into comparable groups for price comparison:
1. Load and validate normalized product data
2. Configure algorithm parameters
3. Pass 1: Branded product matching (same known brand across supermarkets)
4. Pass 2: Own-brand tier matching (same tier, e.g. Value ↔ Value)
5. Pass 3: Unbranded third-party matching (name similarity only)
6. Union-Find cluster assembly
7. Post-process violations (same-supermarket, cross-tier)
8. Save output files
9. Generate stratified audit sample (50 clusters)
10. Diagnostic statistics and coverage report

In [1]:
import pandas as pd
import numpy as np
import os
import json
import random
import warnings
from collections import defaultdict
from itertools import combinations
from tqdm import tqdm
from rapidfuzz import fuzz

warnings.filterwarnings('ignore')

# ============================================================
# CONFIGURATION
# ============================================================

# Unit value tolerance (fraction) — how much weight can differ between matched products
UNIT_TOLERANCE_BRANDED    = 0.05   # ±5% for branded (e.g. 415g ↔ 415g, or 395g)
UNIT_TOLERANCE_OWN_BRAND  = 0.03   # ±3% for own-brand (tier match must be tight)
UNIT_TOLERANCE_UNBRANDED  = 0.05   # ±5% for unbranded

# Fuzzy name similarity thresholds (0–1 scale)
FUZZY_THRESHOLD           = 0.82   # General threshold
FUZZY_THRESHOLD_NOUNIT    = 0.88   # Higher threshold when no unit_value available

# Pack quantity: if ratio between two pack_quantity values exceeds this, reject
PACK_QTY_MAX_RATIO        = 4.0

# Max products per block before sub-blocking kicks in
MAX_BLOCK_SIZE            = 200

# Attribute penalties: applied to fuzzy score when attributes mismatch
ATTR_PENALTIES = {
    'organic':   0.20,
    'free_from': 0.20,
    'fairtrade': 0.10,
    'diet':      0.05,
}

# Known-brand extraction errors — route these as unbranded instead
BRAND_EXCLUSIONS = {'extra', 'essential', 'basics', 'finest', 'select', 'special'}

# Output directory
OUTPUT_DIR = 'data/clusters'

# Reproducibility
RANDOM_SEED = 42

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print('Configuration loaded.')
print(f'  Fuzzy threshold:       {FUZZY_THRESHOLD}')
print(f'  Unit tolerance:        ±{UNIT_TOLERANCE_BRANDED*100:.0f}% (branded), ±{UNIT_TOLERANCE_OWN_BRAND*100:.0f}% (own-brand)')
print(f'  Max block size:        {MAX_BLOCK_SIZE}')
print(f'  Output directory:      {OUTPUT_DIR}')

Configuration loaded.
  Fuzzy threshold:       0.82
  Unit tolerance:        ±5% (branded), ±3% (own-brand)
  Max block size:        200
  Output directory:      data/clusters


In [2]:
# ============================================================
# CELL 3: Data Loading and Validation
# ============================================================

df = pd.read_csv('data/normalized_products.csv', low_memory=False)
print(f'Loaded {len(df):,} products, {df.shape[1]} columns')

# Validate required columns
REQUIRED_COLS = [
    'supermarket', 'names', 'category', 'own_brand',
    'supermarket_brand', 'tier_type', 'known_brand',
    'pack_quantity', 'unit_value', 'unit_type',
    'attributes_keywords', 'core_product_name', 'normalized_name'
]
missing = [c for c in REQUIRED_COLS if c not in df.columns]
assert not missing, f'Missing columns: {missing}'

# Assign product index (used by Union-Find)
df = df.reset_index(drop=True)
df['product_idx'] = df.index

# Clean up known_brand — exclude extraction errors
df['known_brand_clean'] = df['known_brand'].where(
    ~df['known_brand'].str.lower().isin(BRAND_EXCLUSIONS), other=None
)

# Derive product_type column
df['product_type'] = np.where(
    df['known_brand_clean'].notna(), 'branded',
    np.where(df['own_brand'].astype(str).str.lower().isin(['true', '1', 'yes']), 'own_brand', 'unbranded')
)

# Derive helper booleans
df['has_unit'] = df['unit_value'].notna() & (df['unit_value'] > 0)
df['has_pack'] = df['pack_quantity'].notna() & (df['pack_quantity'] > 0)

# Normalize supermarket names
df['supermarket'] = df['supermarket'].str.strip()

# ---- Summary ----
print(f'\nSupermarket distribution:')
print(df['supermarket'].value_counts().to_string())

print(f'\nProduct type breakdown:')
print(df['product_type'].value_counts().to_string())

print(f'\nUnit coverage:')
for pt in ['branded', 'own_brand', 'unbranded']:
    sub = df[df['product_type'] == pt]
    pct = sub['has_unit'].mean() * 100
    print(f'  {pt:12s}: {pct:.1f}% have unit_value  (n={len(sub):,})')

print(f'\nNull rates for key fields:')
for col in ['unit_value', 'unit_type', 'pack_quantity', 'tier_type', 'known_brand_clean', 'normalized_name']:
    null_pct = df[col].isna().mean() * 100
    print(f'  {col:20s}: {null_pct:.1f}% null')

Loaded 65,472 products, 24 columns

Supermarket distribution:
supermarket
Sains        17974
ASDA         17205
Tesco        15831
Morrisons    14462

Product type breakdown:
product_type
branded      22557
unbranded    22462
own_brand    20453

Unit coverage:
  branded     : 55.3% have unit_value  (n=22,557)
  own_brand   : 53.6% have unit_value  (n=20,453)
  unbranded   : 56.4% have unit_value  (n=22,462)

Null rates for key fields:
  unit_value          : 44.8% null
  unit_type           : 44.8% null
  pack_quantity       : 87.9% null
  tier_type           : 71.5% null
  known_brand_clean   : 65.5% null
  normalized_name     : 0.0% null


In [3]:
# ============================================================
# CELL 4: Union-Find Data Structure
# ============================================================

class UnionFind:
    """Disjoint Set Union with path compression and union by rank."""

    def __init__(self, n: int):
        self.parent = list(range(n))
        self.rank = [0] * n

    def find(self, x: int) -> int:
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]  # path compression
            x = self.parent[x]
        return x

    def union(self, x: int, y: int) -> bool:
        px, py = self.find(x), self.find(y)
        if px == py:
            return False
        if self.rank[px] < self.rank[py]:
            px, py = py, px
        self.parent[py] = px
        if self.rank[px] == self.rank[py]:
            self.rank[px] += 1
        return True

    def components(self) -> dict:
        """Return dict mapping root -> list of indices in that component."""
        comps = defaultdict(list)
        for i in range(len(self.parent)):
            comps[self.find(i)].append(i)
        return dict(comps)


print('UnionFind class ready.')

UnionFind class ready.


In [4]:
# ============================================================
# CELL 5: Similarity Function
# ============================================================

def _unit_value_compatible(uv_a, uv_b, tolerance: float) -> bool:
    """Both unit values must be within `tolerance` fraction of each other."""
    if pd.isna(uv_a) or pd.isna(uv_b) or uv_a <= 0 or uv_b <= 0:
        return True  # no constraint when unit_value is missing
    larger = max(uv_a, uv_b)
    return abs(uv_a - uv_b) / larger <= tolerance


def _unit_type_compatible(ut_a, ut_b) -> bool:
    """Unit types (g, ml, unit) must match if both are set."""
    if pd.isna(ut_a) or pd.isna(ut_b):
        return True
    return str(ut_a).strip() == str(ut_b).strip()


def _pack_compatible(pq_a, pq_b) -> bool:
    """Pack quantities must not differ by more than PACK_QTY_MAX_RATIO."""
    if pd.isna(pq_a) or pd.isna(pq_b) or pq_a <= 0 or pq_b <= 0:
        return True
    ratio = max(pq_a, pq_b) / min(pq_a, pq_b)
    return ratio <= PACK_QTY_MAX_RATIO


def _attribute_penalty(attrs_a, attrs_b) -> float:
    """Return a penalty [0, 1] based on mismatched defining attributes."""
    penalty = 0.0
    if pd.isna(attrs_a):
        attrs_a = ''
    if pd.isna(attrs_b):
        attrs_b = ''
    str_a, str_b = str(attrs_a).lower(), str(attrs_b).lower()
    for attr, p in ATTR_PENALTIES.items():
        has_a = attr in str_a
        has_b = attr in str_b
        if has_a != has_b:
            penalty += p
    return min(penalty, 1.0)


def compute_similarity(
    row_a: pd.Series,
    row_b: pd.Series,
    pass_type: str,
    unit_tolerance: float,
) -> tuple:
    """
    Returns (is_match: bool, score: float).
    
    Hard constraints are checked first (cheapest, most eliminating).
    Only reaches fuzzy matching if all hard constraints pass.
    """
    # Hard constraint 1: same supermarket → never match
    if row_a['supermarket'] == row_b['supermarket']:
        return False, 0.0

    # Hard constraint 2: unit type must be compatible
    if not _unit_type_compatible(row_a.get('unit_type'), row_b.get('unit_type')):
        return False, 0.0

    # Hard constraint 3: unit value must be within tolerance
    if not _unit_value_compatible(row_a.get('unit_value'), row_b.get('unit_value'), unit_tolerance):
        return False, 0.0

    # Hard constraint 4: pack quantity ratio
    if not _pack_compatible(row_a.get('pack_quantity'), row_b.get('pack_quantity')):
        return False, 0.0

    # Hard constraint 5: brand / tier rules per pass type
    if pass_type == 'branded':
        # Must be same brand
        if str(row_a['known_brand_clean']).lower() != str(row_b['known_brand_clean']).lower():
            return False, 0.0

    elif pass_type == 'own_brand':
        # Must be same tier type
        tier_a = str(row_a.get('tier_type', '')).lower() if pd.notna(row_a.get('tier_type')) else 'standard'
        tier_b = str(row_b.get('tier_type', '')).lower() if pd.notna(row_b.get('tier_type')) else 'standard'
        if tier_a != tier_b:
            return False, 0.0
        # Own-brand must not match branded products
        if row_a.get('product_type') != row_b.get('product_type'):
            return False, 0.0

    # Short names (≤2 tokens) require unit_value present on both sides
    name_a = str(row_a.get('normalized_name', '') or '')
    name_b = str(row_b.get('normalized_name', '') or '')
    if len(name_a.split()) <= 2 or len(name_b.split()) <= 2:
        if not (row_a['has_unit'] and row_b['has_unit']):
            return False, 0.0

    # Fuzzy name similarity
    token_set = fuzz.token_set_ratio(name_a, name_b) / 100.0
    partial   = fuzz.partial_ratio(name_a, name_b) / 100.0
    score = 0.7 * token_set + 0.3 * partial

    # Attribute penalty
    penalty = _attribute_penalty(
        row_a.get('attributes_keywords'), row_b.get('attributes_keywords')
    )
    score -= penalty

    # Apply appropriate threshold
    threshold = FUZZY_THRESHOLD_NOUNIT if (not row_a['has_unit'] or not row_b['has_unit']) else FUZZY_THRESHOLD

    return score >= threshold, score


# ---- Self-test ----
print('Running similarity self-tests...')

_test_cases = [
    # (description, name_a, name_b, unit_a, unit_b, ut_a, ut_b, sm_a, sm_b, expected_match)
    ('Same product different SM',       'baked beans in tomato sauce', 'baked beans tomato sauce', 415, 415, 'g', 'g', 'Tesco', 'ASDA',       True),
    ('Different product same weight',   'plum tomatoes',               'chopped tomatoes',         400, 400, 'g', 'g', 'Tesco', 'ASDA',       False),
    ('Same supermarket',                'whole milk',                  'whole milk',               2270, 2270, 'g', 'g', 'Tesco', 'Tesco',     False),
    ('Weight mismatch (200g vs 400g)',  'sweetcorn in water',          'sweetcorn in water',       200, 400, 'g', 'g', 'Tesco', 'ASDA',       False),
    ('Unit type mismatch g vs ml',      'orange juice',                'orange juice',             1000, 1000, 'g', 'ml', 'Tesco', 'ASDA',    False),
]

all_pass = True
for desc, na, nb, uva, uvb, uta, utb, sma, smb, expected in _test_cases:
    ra = pd.Series({'normalized_name': na, 'unit_value': uva, 'unit_type': uta, 'supermarket': sma,
                    'has_unit': True, 'pack_quantity': None, 'attributes_keywords': None,
                    'known_brand_clean': 'heinz', 'tier_type': 'standard', 'product_type': 'branded'})
    rb = pd.Series({'normalized_name': nb, 'unit_value': uvb, 'unit_type': utb, 'supermarket': smb,
                    'has_unit': True, 'pack_quantity': None, 'attributes_keywords': None,
                    'known_brand_clean': 'heinz', 'tier_type': 'standard', 'product_type': 'branded'})
    match, score = compute_similarity(ra, rb, 'branded', UNIT_TOLERANCE_BRANDED)
    status = '✓' if match == expected else '✗'
    if match != expected:
        all_pass = False
    print(f'  {status} {desc}: match={match}, score={score:.3f}')

print(f'\nAll tests passed: {all_pass}')

Running similarity self-tests...
  ✓ Same product different SM: match=True, score=0.962
  ✓ Different product same weight: match=False, score=0.779
  ✓ Same supermarket: match=False, score=0.000
  ✓ Weight mismatch (200g vs 400g): match=False, score=0.000
  ✓ Unit type mismatch g vs ml: match=False, score=0.000

All tests passed: True


In [5]:
# ============================================================
# CELL 6: Blocking Infrastructure
# ============================================================

STOPWORDS = {'the', 'a', 'an', 'of', 'and', 'with', 'in', 'for', 'to', 'by', '&', 'or', 'no'}

def _unit_bucket(uv, bucket_size=50):
    """Round unit_value to nearest bucket_size for sub-blocking."""
    if pd.isna(uv) or uv <= 0:
        return 'no_unit'
    return str(round(float(uv) / bucket_size) * bucket_size)


def build_blocks(sub_df: pd.DataFrame, key_fn, max_block_size: int = MAX_BLOCK_SIZE):
    """
    Group products by block key.
    Only keeps blocks containing products from 2+ different supermarkets.
    Sub-blocks oversized blocks by unit_value_bucket.
    Returns list of (block_key, list_of_product_indices).
    """
    # Build initial blocks
    raw_blocks = defaultdict(list)
    for idx, row in sub_df.iterrows():
        key = key_fn(row)
        raw_blocks[key].append(idx)

    blocks = []
    total_pairs = 0

    for key, indices in raw_blocks.items():
        block_df = sub_df.loc[indices]

        # Skip blocks with only 1 supermarket (nothing to cross-match)
        if block_df['supermarket'].nunique() < 2:
            continue

        # Sub-block if too large
        if len(indices) > max_block_size:
            sub_raw = defaultdict(list)
            for idx in indices:
                sub_key = key + '||' + _unit_bucket(sub_df.loc[idx, 'unit_value'])
                sub_raw[sub_key].append(idx)
            for sub_key, sub_idx in sub_raw.items():
                sub_block_df = sub_df.loc[sub_idx]
                if sub_block_df['supermarket'].nunique() < 2:
                    continue
                blocks.append((sub_key, sub_idx))
                n = len(sub_idx)
                total_pairs += n * (n - 1) // 2
        else:
            blocks.append((key, indices))
            n = len(indices)
            total_pairs += n * (n - 1) // 2

    block_sizes = [len(b[1]) for b in blocks]
    print(f'  Blocks: {len(blocks):,}  |  Total pairs: {total_pairs:,}')
    if block_sizes:
        print(f'  Block size — mean: {np.mean(block_sizes):.1f}, max: {max(block_sizes)}, median: {int(np.median(block_sizes))}')

    return blocks


def run_pass(
    sub_df: pd.DataFrame,
    blocks: list,
    pass_type: str,
    unit_tolerance: float,
) -> list:
    """
    Compare all pairs within each block.
    Returns list of (idx_a, idx_b, score) match tuples.
    """
    matches = []
    for _key, indices in tqdm(blocks, desc=f'  Pass={pass_type}', leave=True):
        for ia, ib in combinations(indices, 2):
            row_a = sub_df.loc[ia]
            row_b = sub_df.loc[ib]
            is_match, score = compute_similarity(row_a, row_b, pass_type, unit_tolerance)
            if is_match:
                matches.append((ia, ib, score))
    return matches


print('Blocking infrastructure ready.')

Blocking infrastructure ready.


In [6]:
# ============================================================
# CELL 7: Pass 1 — Branded Product Matching
# ============================================================
# Block on: known_brand + category + unit_type
# Hard rule: same known_brand required

branded_df = df[df['product_type'] == 'branded'].copy()
print(f'Pass 1 — Branded: {len(branded_df):,} products')

def branded_block_key(row):
    brand = str(row['known_brand_clean']).lower().strip()
    cat   = str(row['category']).lower().strip() if pd.notna(row['category']) else 'unknown'
    utype = str(row['unit_type']).strip() if pd.notna(row['unit_type']) else 'none'
    return f'{brand}||{cat}||{utype}'

print('\nBuilding blocks...')
blocks_branded = build_blocks(branded_df, branded_block_key)

print('\nRunning comparisons...')
matches_branded = run_pass(branded_df, blocks_branded, 'branded', UNIT_TOLERANCE_BRANDED)
print(f'\nPass 1 complete: {len(matches_branded):,} matches found')

Pass 1 — Branded: 22,557 products

Building blocks...
  Blocks: 959  |  Total pairs: 564,524
  Block size — mean: 20.9, max: 307, median: 12

Running comparisons...


  Pass=branded: 100%|██████████| 959/959 [00:39<00:00, 24.14it/s] 


Pass 1 complete: 15,688 matches found


In [7]:
# ============================================================
# CELL 8: Pass 2 — Own-Brand Tier Matching
# ============================================================
# Block on: tier_type + category + unit_type + first_significant_token
# Hard rule: same tier_type required

own_brand_df = df[df['product_type'] == 'own_brand'].copy()
print(f'Pass 2 — Own-brand: {len(own_brand_df):,} products')

def own_brand_block_key(row):
    tier  = str(row['tier_type']).lower().strip() if pd.notna(row['tier_type']) else 'standard'
    cat   = str(row['category']).lower().strip() if pd.notna(row['category']) else 'unknown'
    utype = str(row['unit_type']).strip() if pd.notna(row['unit_type']) else 'none'
    norm  = str(row['normalized_name'] or '').lower()
    tokens = [t for t in norm.split() if t not in STOPWORDS]
    first_tok = tokens[0] if tokens else 'unknown'
    return f'{tier}||{cat}||{utype}||{first_tok}'

print('\nBuilding blocks...')
blocks_own = build_blocks(own_brand_df, own_brand_block_key)

print('\nRunning comparisons...')
matches_own = run_pass(own_brand_df, blocks_own, 'own_brand', UNIT_TOLERANCE_OWN_BRAND)
print(f'\nPass 2 complete: {len(matches_own):,} matches found')

Pass 2 — Own-brand: 20,453 products

Building blocks...
  Blocks: 1,752  |  Total pairs: 146,101
  Block size — mean: 6.9, max: 268, median: 4

Running comparisons...


  Pass=own_brand: 100%|██████████| 1752/1752 [00:09<00:00, 176.42it/s]


Pass 2 complete: 2,384 matches found


In [8]:
# ============================================================
# CELL 9: Pass 3 — Unbranded Third-Party Matching
# ============================================================
# Block on: category + unit_type + sorted first two significant tokens
# No brand constraint — relies on name similarity alone

unbranded_df = df[df['product_type'] == 'unbranded'].copy()
print(f'Pass 3 — Unbranded: {len(unbranded_df):,} products')

def unbranded_block_key(row):
    cat   = str(row['category']).lower().strip() if pd.notna(row['category']) else 'unknown'
    utype = str(row['unit_type']).strip() if pd.notna(row['unit_type']) else 'none'
    norm  = str(row['normalized_name'] or '').lower()
    tokens = [t for t in norm.split() if t not in STOPWORDS]
    # Sort first 2 significant tokens for order-independence
    sig_tokens = sorted(tokens[:2])
    tok_key = '_'.join(sig_tokens) if sig_tokens else 'unknown'
    return f'{cat}||{utype}||{tok_key}'

print('\nBuilding blocks...')
blocks_unbranded = build_blocks(unbranded_df, unbranded_block_key)

print('\nRunning comparisons...')
matches_unbranded = run_pass(unbranded_df, blocks_unbranded, 'unbranded', UNIT_TOLERANCE_UNBRANDED)
print(f'\nPass 3 complete: {len(matches_unbranded):,} matches found')

Pass 3 — Unbranded: 22,462 products

Building blocks...
  Blocks: 1,756  |  Total pairs: 49,646
  Block size — mean: 5.2, max: 66, median: 3

Running comparisons...


  Pass=unbranded: 100%|██████████| 1756/1756 [00:03<00:00, 505.60it/s] 


Pass 3 complete: 4,679 matches found


In [9]:
# ============================================================
# CELL 10: Union-Find Assembly
# ============================================================

print('Assembling Union-Find from all passes...')
uf = UnionFind(len(df))

# Track match scores for later averaging
pair_scores: dict = {}  # (min_idx, max_idx) -> score

all_matches = (
    [('branded', m) for m in matches_branded] +
    [('own_brand', m) for m in matches_own] +
    [('unbranded', m) for m in matches_unbranded]
)

for _pass, (ia, ib, score) in all_matches:
    uf.union(ia, ib)
    pair_scores[(min(ia, ib), max(ia, ib))] = score

# Assign cluster IDs
df['raw_cluster_id'] = df['product_idx'].apply(uf.find)

# Cluster size
cluster_sizes = df.groupby('raw_cluster_id')['product_idx'].count()
df['cluster_size_raw'] = df['raw_cluster_id'].map(cluster_sizes)

n_clusters_raw = df['raw_cluster_id'].nunique()
n_singletons_raw = (cluster_sizes == 1).sum()
n_multi_raw = (cluster_sizes > 1).sum()

print(f'\n--- Raw Union-Find Results ---')
print(f'Total clusters:     {n_clusters_raw:,}')
print(f'Singletons:         {n_singletons_raw:,}')
print(f'Multi-product:      {n_multi_raw:,}')
print(f'Largest cluster:    {cluster_sizes.max()}')

size_dist = cluster_sizes.value_counts().sort_index()
print(f'\nCluster size distribution (raw):')
for size, count in size_dist.items():
    if size <= 8:
        print(f'  size {size}: {count:,} clusters')
    elif size == size_dist.index.max():
        print(f'  ...max {size}: {count:,} clusters')

Assembling Union-Find from all passes...

--- Raw Union-Find Results ---
Total clusters:     50,650
Singletons:         44,312
Multi-product:      6,338
Largest cluster:    156

Cluster size distribution (raw):
  size 1: 44,312 clusters
  size 2: 3,989 clusters
  size 3: 961 clusters
  size 4: 512 clusters
  size 5: 239 clusters
  size 6: 174 clusters
  size 7: 106 clusters
  size 8: 64 clusters
  ...max 156: 1 clusters


In [10]:
# ============================================================
# CELL 11: Post-Processing — Fix Violations
# ============================================================

def fix_same_supermarket_violation(group_df: pd.DataFrame, pair_scores: dict) -> list:
    """
    If a cluster has >1 product from the same supermarket,
    split into sub-clusters of at most 1 product per SM.
    Uses a greedy approach: build clusters by adding products one at a time,
    skipping if the supermarket already has a representative.
    Returns list of DataFrames, each representing a valid sub-cluster.
    """
    by_sm = group_df.groupby('supermarket')
    problem_sms = [sm for sm, g in by_sm if len(g) > 1]

    if not problem_sms:
        return [group_df]  # no violation

    # Greedy assignment: pick one representative per SM per sub-cluster
    # Sort products by average pairwise score descending (best matches first)
    sub_clusters = []
    remaining = group_df.copy()

    while len(remaining) > 0:
        sub = []
        seen_sms = set()
        for idx, row in remaining.iterrows():
            sm = row['supermarket']
            if sm not in seen_sms:
                sub.append(idx)
                seen_sms.add(sm)
        sub_df = remaining.loc[sub]
        sub_clusters.append(sub_df)
        remaining = remaining.drop(sub)

    return sub_clusters


def fix_cross_tier_violation(group_df: pd.DataFrame) -> list:
    """
    For own-brand clusters, split by tier_type.
    Returns list of DataFrames (one per tier present).
    """
    own = group_df[group_df['product_type'] == 'own_brand']
    if own.empty or own['tier_type'].nunique() <= 1:
        return [group_df]

    sub_clusters = []
    for tier, tier_group in own.groupby('tier_type'):
        # Also include any branded/unbranded members with the cluster
        non_own = group_df[group_df['product_type'] != 'own_brand']
        sub_clusters.append(pd.concat([tier_group, non_own]))

    return sub_clusters


# ---- Run post-processing ----
print('Running post-processing...')

final_clusters = []   # list of DataFrames, each a valid cluster
sm_violations_fixed = 0
tier_violations_fixed = 0

for raw_cid, group in tqdm(df.groupby('raw_cluster_id'), desc='Post-processing'):
    if len(group) == 1:
        # Singleton — keep as-is
        final_clusters.append(group)
        continue

    # Step 1: Fix same-SM violation
    sub_clusters = fix_same_supermarket_violation(group, pair_scores)
    if len(sub_clusters) > 1:
        sm_violations_fixed += 1

    # Step 2: Fix cross-tier violation within each sub-cluster
    for sc in sub_clusters:
        tier_subs = fix_cross_tier_violation(sc)
        if len(tier_subs) > 1:
            tier_violations_fixed += 1
        final_clusters.extend(tier_subs)

print(f'\nPost-processing complete:')
print(f'  Same-SM violations fixed:    {sm_violations_fixed:,}')
print(f'  Cross-tier violations fixed: {tier_violations_fixed:,}')
print(f'  Final cluster count:         {len(final_clusters):,}')

# Assign sequential cluster IDs (largest clusters first)
final_clusters.sort(key=lambda g: -len(g))
cluster_id_map = {}  # product_idx -> cluster_id
match_type_map = {}  # cluster_id -> match_type

for cid, group in enumerate(final_clusters):
    for idx in group.index:
        cluster_id_map[idx] = cid
    # Determine match type from product types in cluster
    types = group['product_type'].value_counts()
    match_type_map[cid] = types.index[0] if len(types) else 'unknown'

df['cluster_id'] = df.index.map(cluster_id_map)

print(f'\nFinal cluster size distribution:')
final_sizes = df.groupby('cluster_id')['product_idx'].count()
size_dist_final = final_sizes.value_counts().sort_index()
for size, count in size_dist_final.items():
    if size <= 6:
        print(f'  size {size}: {count:,}')

Running post-processing...


Post-processing: 100%|██████████| 50650/50650 [00:09<00:00, 5240.47it/s] 



Post-processing complete:
  Same-SM violations fixed:    2,081
  Cross-tier violations fixed: 0
  Final cluster count:         55,497

Final cluster size distribution:
  size 1: 46,465
  size 2: 8,171
  size 3: 779
  size 4: 82


In [11]:
# ============================================================
# CELL 12: Output Generation
# ============================================================

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ---- Compute avg pairwise score per cluster ----
cluster_avg_scores = defaultdict(list)
cluster_min_scores = defaultdict(list)

for (ia, ib), score in pair_scores.items():
    cid_a = cluster_id_map.get(ia)
    cid_b = cluster_id_map.get(ib)
    if cid_a == cid_b and cid_a is not None:
        cluster_avg_scores[cid_a].append(score)

avg_score_map = {cid: np.mean(scores) for cid, scores in cluster_avg_scores.items()}
min_score_map = {cid: np.min(scores) for cid, scores in cluster_avg_scores.items()}

# ---- clusters.csv — one row per product ----
clusters_df = df[[
    'cluster_id', 'product_idx', 'supermarket', 'names', 'category',
    'known_brand_clean', 'own_brand', 'tier_type', 'unit_value', 'unit_type',
    'pack_quantity', 'core_product_name', 'normalized_name',
    'prices_(£)', 'prices_unit_(£)', 'product_type'
]].copy()

cluster_size_series = df.groupby('cluster_id')['product_idx'].transform('count')
clusters_df['cluster_size'] = cluster_size_series
clusters_df['n_supermarkets'] = df.groupby('cluster_id')['supermarket'].transform('nunique')
clusters_df['match_type'] = clusters_df['cluster_id'].map(match_type_map)
clusters_df['avg_pairwise_score'] = clusters_df['cluster_id'].map(avg_score_map)

clusters_path = os.path.join(OUTPUT_DIR, 'clusters.csv')
clusters_df.to_csv(clusters_path, index=False)
print(f'Saved: {clusters_path}  ({len(clusters_df):,} rows)')

# ---- cluster_summary.csv — one row per cluster ----
summary_rows = []
for cid, group in tqdm(clusters_df.groupby('cluster_id'), desc='Building summary'):
    # Consensus name: shortest core_product_name in cluster
    names_avail = group['core_product_name'].dropna()
    consensus_name = names_avail.loc[names_avail.str.len().idxmin()] if len(names_avail) else ''

    summary_rows.append({
        'cluster_id':              cid,
        'cluster_size':            len(group),
        'n_supermarkets':          group['supermarket'].nunique(),
        'supermarkets_present':    '|'.join(sorted(group['supermarket'].unique())),
        'category':                group['category'].mode()[0] if len(group) else None,
        'match_type':              match_type_map.get(cid, 'unknown'),
        'known_brand':             group['known_brand_clean'].dropna().iloc[0] if group['known_brand_clean'].notna().any() else None,
        'tier_type':               group['tier_type'].dropna().iloc[0] if group['tier_type'].notna().any() else None,
        'unit_value':              group['unit_value'].dropna().mean() if group['unit_value'].notna().any() else None,
        'unit_type':               group['unit_type'].dropna().iloc[0] if group['unit_type'].notna().any() else None,
        'pack_quantity':           group['pack_quantity'].dropna().mean() if group['pack_quantity'].notna().any() else None,
        'core_product_name_consensus': consensus_name,
        'avg_pairwise_score':      avg_score_map.get(cid),
        'min_pairwise_score':      min_score_map.get(cid),
    })

cluster_summary = pd.DataFrame(summary_rows)
summary_path = os.path.join(OUTPUT_DIR, 'cluster_summary.csv')
cluster_summary.to_csv(summary_path, index=False)
print(f'Saved: {summary_path}  ({len(cluster_summary):,} rows)')

# ---- singletons.csv — products that matched nothing ----
singletons = clusters_df[clusters_df['cluster_size'] == 1].copy()
singletons_path = os.path.join(OUTPUT_DIR, 'singletons.csv')
singletons.to_csv(singletons_path, index=False)
print(f'Saved: {singletons_path}  ({len(singletons):,} rows)')

Saved: data/clusters/clusters.csv  (65,472 rows)


Building summary: 100%|██████████| 55497/55497 [00:24<00:00, 2245.19it/s]


Saved: data/clusters/cluster_summary.csv  (55,497 rows)
Saved: data/clusters/singletons.csv  (46,465 rows)


In [12]:
# ============================================================
# CELL 13: Audit Sample Generation
# ============================================================
# Stratified 50-cluster sample: 20 branded + 20 own-brand + 10 unbranded
# Only clusters with ≥2 products (something to compare)

multi_clusters = cluster_summary[cluster_summary['cluster_size'] >= 2].copy()

branded_pool   = multi_clusters[multi_clusters['match_type'] == 'branded']
own_pool       = multi_clusters[multi_clusters['match_type'] == 'own_brand']
unbranded_pool = multi_clusters[multi_clusters['match_type'] == 'unbranded']

n_branded   = min(20, len(branded_pool))
n_own       = min(20, len(own_pool))
n_unbranded = min(10, len(unbranded_pool))

sampled_branded   = branded_pool.sample(n_branded, random_state=RANDOM_SEED)
sampled_own       = own_pool.sample(n_own, random_state=RANDOM_SEED)
sampled_unbranded = unbranded_pool.sample(n_unbranded, random_state=RANDOM_SEED)

audit_ids = pd.concat([
    sampled_branded, sampled_own, sampled_unbranded
])['cluster_id'].tolist()

audit_df = clusters_df[clusters_df['cluster_id'].isin(audit_ids)].copy()
audit_df = audit_df.sort_values(['cluster_id', 'supermarket'])

# Add audit columns for the reviewer
audit_df['AUDIT_same_core_product'] = ''   # Reviewer fills: Yes/No
audit_df['AUDIT_weight_ok']         = ''   # Reviewer fills: Yes/No
audit_df['AUDIT_tier_ok']           = ''   # Reviewer fills: Yes/No (N/A if not own-brand)
audit_df['AUDIT_notes']             = ''

audit_path = os.path.join(OUTPUT_DIR, 'audit_sample_50.csv')
audit_df.to_csv(audit_path, index=False)

print(f'Audit sample saved: {audit_path}')
print(f'  Branded clusters:   {n_branded}')
print(f'  Own-brand clusters: {n_own}')
print(f'  Unbranded clusters: {n_unbranded}')
print(f'  Total clusters:     {len(audit_ids)}')
print(f'  Total rows:         {len(audit_df)}')

# Preview a few clusters
print('\n--- Sample clusters (first 3) ---')
for cid in audit_ids[:3]:
    cluster_rows = audit_df[audit_df['cluster_id'] == cid]
    print(f'\nCluster {cid} ({cluster_rows["match_type"].iloc[0]}, {cluster_rows["unit_value"].iloc[0]}g):')
    for _, row in cluster_rows.iterrows():
        print(f'  [{row["supermarket"]:10s}] {row["names"]}')

Audit sample saved: data/clusters/audit_sample_50.csv
  Branded clusters:   20
  Own-brand clusters: 20
  Unbranded clusters: 10
  Total clusters:     50
  Total rows:         106

--- Sample clusters (first 3) ---

Cluster 7405 (branded, 750.0g):
  [Sains     ] 19 Crimes Revolutionary Rosé 750ml
  [Tesco     ] 19 Crimes Red Wine 75Cl

Cluster 5677 (branded, 515.0g):
  [Sains     ] Nestle Honey Cheerios Cereal 515g
  [Tesco     ] Nestlé Cheerios Multigrain Cereal 540g

Cluster 640 (branded, nang):
  [ASDA      ] Mr Kipling Lemon & Raspberry Mini Batts
  [Morrisons ] Mr Kipling Lemon & Raspberry Mini Batts
  [Tesco     ] Mr Kipling Lemon & Raspberry Mini Batts 5 Pack


In [13]:
# ============================================================
# CELL 14: Diagnostic Statistics & Automated Validation
# ============================================================

print('=' * 60)
print('SHOPWISER CLUSTERING — DIAGNOSTIC REPORT')
print('=' * 60)

non_singleton = cluster_summary[cluster_summary['cluster_size'] >= 2]
n_total_clusters = len(cluster_summary)
n_singleton_clusters = len(cluster_summary[cluster_summary['cluster_size'] == 1])
n_multi_clusters = len(non_singleton)

print(f'\n--- Cluster Counts ---')
print(f'Total clusters (incl. singletons):  {n_total_clusters:,}')
print(f'Singleton clusters:                 {n_singleton_clusters:,}')
print(f'Multi-product clusters (≥2):        {n_multi_clusters:,}')
print(f'  4-way (all supermarkets):         {(non_singleton["n_supermarkets"] == 4).sum():,}')
print(f'  3-way:                            {(non_singleton["n_supermarkets"] == 3).sum():,}')
print(f'  2-way:                            {(non_singleton["n_supermarkets"] == 2).sum():,}')

print(f'\n--- By Match Type (multi-product only) ---')
for mt in ['branded', 'own_brand', 'unbranded']:
    sub = non_singleton[non_singleton['match_type'] == mt]
    print(f'  {mt:12s}: {len(sub):,} clusters')

print(f'\n--- Product Coverage ---')
in_cluster = clusters_df[clusters_df['cluster_size'] >= 2]
for sm in sorted(df['supermarket'].unique()):
    total = len(df[df['supermarket'] == sm])
    matched = len(in_cluster[in_cluster['supermarket'] == sm])
    print(f'  {sm:12s}: {matched:,} / {total:,} = {matched/total*100:.1f}% in a cluster')

print(f'\n--- Quality Scores (avg_pairwise_score) ---')
scores_available = non_singleton.dropna(subset=['avg_pairwise_score'])
for mt in ['branded', 'own_brand', 'unbranded']:
    sub = scores_available[scores_available['match_type'] == mt]['avg_pairwise_score']
    if len(sub):
        print(f'  {mt:12s}: mean={sub.mean():.3f}, p5={sub.quantile(0.05):.3f}, p25={sub.quantile(0.25):.3f}')

print(f'\n--- Automated Validation Checks (all {n_multi_clusters:,} multi-clusters) ---')

violations = {
    'same_sm': 0,
    'unit_type_mixed': 0,
    'tier_mixed': 0,
    'weight_variance_high': 0,
    'score_below_075': 0,
}

for cid, group in clusters_df[clusters_df['cluster_size'] >= 2].groupby('cluster_id'):
    # Check 1: no repeated supermarket
    if group['supermarket'].value_counts().max() > 1:
        violations['same_sm'] += 1

    # Check 2: unit_type is consistent
    ut = group['unit_type'].dropna()
    if ut.nunique() > 1:
        violations['unit_type_mixed'] += 1

    # Check 3: own-brand tier consistency
    own = group[group['product_type'] == 'own_brand']
    if len(own) >= 2 and own['tier_type'].nunique() > 1:
        violations['tier_mixed'] += 1

    # Check 4: weight variance ≤ 10%
    uv = group['unit_value'].dropna()
    if len(uv) >= 2 and uv.min() > 0:
        if uv.max() / uv.min() > 1.10:
            violations['weight_variance_high'] += 1

    # Check 5: avg score not too low
    avg_s = avg_score_map.get(cid)
    if avg_s is not None and avg_s < 0.75:
        violations['score_below_075'] += 1

print(f'  Supermarket uniqueness violations:  {violations["same_sm"]:,}  (should be 0)')
print(f'  Mixed unit type:                    {violations["unit_type_mixed"]:,}  (should be 0)')
print(f'  Tier mixing in own-brand clusters:  {violations["tier_mixed"]:,}  (should be 0)')
print(f'  Weight variance > 10%:              {violations["weight_variance_high"]:,}  (flag for review)')
print(f'  Avg score below 0.75:               {violations["score_below_075"]:,}  (flag for review)')

print(f'\n--- Target Check ---')
print(f'Target cluster range: 10,000–20,000')
print(f'Actual multi-clusters: {n_multi_clusters:,}')
in_range = 10_000 <= n_multi_clusters <= 20_000
print(f'In target range: {"YES ✓" if in_range else "NO — review thresholds"}')

print(f'\n--- Output Files ---')
for fname in ['clusters.csv', 'cluster_summary.csv', 'singletons.csv', 'audit_sample_50.csv']:
    fpath = os.path.join(OUTPUT_DIR, fname)
    if os.path.exists(fpath):
        size_mb = os.path.getsize(fpath) / 1e6
        print(f'  ✓ {fname:<30s} ({size_mb:.1f} MB)')
    else:
        print(f'  ✗ {fname} NOT FOUND')

print('\n' + '=' * 60)
print('Run audit_sample_50.csv through manual / LLM review.')
print('Target: ≥45 / 50 clusters pass all 3 checks (90% threshold).')
print('=' * 60)

SHOPWISER CLUSTERING — DIAGNOSTIC REPORT

--- Cluster Counts ---
Total clusters (incl. singletons):  55,497
Singleton clusters:                 46,465
Multi-product clusters (≥2):        9,032
  4-way (all supermarkets):         82
  3-way:                            779
  2-way:                            8,171

--- By Match Type (multi-product only) ---
  branded     : 5,361 clusters
  own_brand   : 1,188 clusters
  unbranded   : 2,483 clusters

--- Product Coverage ---
  ASDA        : 4,644 / 17,205 = 27.0% in a cluster
  Morrisons   : 3,831 / 14,462 = 26.5% in a cluster
  Sains       : 5,265 / 17,974 = 29.3% in a cluster
  Tesco       : 5,267 / 15,831 = 33.3% in a cluster

--- Quality Scores (avg_pairwise_score) ---
  branded     : mean=0.947, p5=0.841, p25=0.908
  own_brand   : mean=0.967, p5=0.850, p25=0.942
  unbranded   : mean=0.958, p5=0.848, p25=0.925

--- Automated Validation Checks (all 9,032 multi-clusters) ---
  Supermarket uniqueness violations:  0  (should be 0)
  Mixed